In [1]:
## In this file, we will be processing the stock summary from the data we have fetched and analysied using LLM models.

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 

In [7]:
## Importer

import os
import time
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from langchain_groq import ChatGroq

## Fetching credentials from env for DataBase
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

api_key = os.getenv("GOOGLE_API_KEY")

db_url = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}' 
engine = create_engine(db_url)

## API Key
os.environ["GOOGLE_API_KEY"] = api_key

print("Credentials fetched")

Credentials fetched


In [10]:
## Using temperature 0.1 for LLM to add its own narrative a bit.
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)

query = """
        select 
            d.asset_id,d.ticker,d.company_name,d.sector,m.current_price,
            coalesce(f.peg_ratio,999) as peg_ratio,
            coalesce(f.roe_percent,0) as roe_percent,
            coalesce(f.debt_to_equity,999) as debt_to_equity,
            COALESCE(f.earnings_growth, 0) as earnings_growth,
            COALESCE(f.gross_profits, 0) as gross_profits,
            coalesce(s.avg_senti_score,0) as sentiment,
            m.momentum_signal
        from sample_set_100 d
        left join fact_fundamentals f on d.asset_id = f.asset_id
        left join stock_momentum m on d.asset_id = m.asset_id
        left join (
                    select asset_id,avg(sentiment_score) as avg_senti_score
                    from fact_news n
                    group by asset_id
                ) s on d.asset_id = s.asset_id;
"""
with engine.connect() as conn:
    df = pd.read_sql(query,conn)

print(f"Data loaded for {len(df)} stocks, AI summary generation starting...")

insights_collector = []

for index,row in df.iterrows():
    prompt = f"""
                You are a quantitative hedge fund analyst. Write a strict, 
                3-sentence executive summary for {row['company_name']} ({row['ticker']}) in the {row['sector']} sector.

                Data context:
                    - Price & Momentum: {row['current_price']} ({row['momentum_signal']})
                    - Valuation: PEG Ratio {row['peg_ratio']} (Note: < 1 is undervalued. 999 means missing/negative data).
                    - Profitability & Growth: ROE {row['roe_percent']}, Earnings Growth {row['earnings_growth']}
                    - Financial Stability: Debt-to-Equity {row['debt_to_equity']} (Note: > 2.0 is high risk. 999 means missing/assumed high risk).
                    - News Sentiment: {row['sentiment']} (Note: -1 to 1 scale. 0 means neutral or no news).

                 Rules:
                    1. Be brutally objective. No fluff. Don't sugarcoat the answer.
                    2. Structure your response perfectly:
                        - Sentence 1: Valuation & Momentum setup.
                        - Sentence 2: Financial stability & growth analysis (use debt and earnings).
                        - Sentence 3: Sentiment context and your final cutthroat verdict.
                    3. Do NOT include any financial advice disclaimers. Just the analysis.
            """
    try:
        response = llm.invoke(prompt)
        ai_ans= response.content.strip()

        insights_collector.append({
            "asset_id" : row["asset_id"],
            "ai_summary" : ai_ans
        })
        print(f"[{index+1}/{len(df)}] Generated insight for {row['ticker']}")

    except Exception as e:
        print(f"Failed for {row["ticker"]}: {e}")

        insights_collector.append({
            'asset_id': row['asset_id'],
            'ai_summary': "AI insight unavailable due to API limits."
        })

    time.sleep(1)

## Push the data to out database
insights_df = pd.DataFrame(insights_collector)
insights_df.to_sql("fact_ai_insights",engine,if_exists="replace",index=False)

print("\n All the AI insights are pushed to out database")


Data loaded for 100 stocks, AI summary generation starting...
[1/100] Generated insight for RELIANCE.NS
[2/100] Generated insight for HDFCBANK.NS
[3/100] Generated insight for BHARTIARTL.NS
[4/100] Generated insight for ICICIBANK.NS
[5/100] Generated insight for SBIN.NS
[6/100] Generated insight for BAJFINANCE.NS
[7/100] Generated insight for INFY.NS
[8/100] Generated insight for HINDUNILVR.NS
[9/100] Generated insight for LICI.NS
[10/100] Generated insight for LT.NS
[11/100] Generated insight for HCLTECH.NS
[12/100] Generated insight for M&M.NS
[13/100] Generated insight for KOTAKBANK.NS
[14/100] Generated insight for AXISBANK.NS
[15/100] Generated insight for SUNPHARMA.NS
[16/100] Generated insight for ADANIPORTS.NS
[17/100] Generated insight for JSWSTEEL.NS
[18/100] Generated insight for ADANIPOWER.NS
[19/100] Generated insight for ADANIENT.NS
[20/100] Generated insight for HAL.NS
[21/100] Generated insight for VEDL.NS
[22/100] Generated insight for ETERNAL.NS
[23/100] Generated ins